# Typhoon 3.5B Distilled - Application Demo
This notebook provides a lightweight demo to interact with the distilled 3.5B model.
It streams tokens with `TextStreamer` and loads from the latest checkpoint subfolder.

In [ ]:
!pip install -q transformers torch accelerate gradio huggingface_hub

In [ ]:
import os
from huggingface_hub import login

# Please replace with your Hugging Face Token
HF_TOKEN = "YOUR_HUGGING_FACE_TOKEN_HERE"
login(token=HF_TOKEN)

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

repo_id = "Phonsiri/typhoon-3.5b-cpt-ckpt"
# Update this to the latest checkpoint step available on HF Hub
STEP = "step_0000275"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(repo_id, subfolder=STEP)

print("Loading model (bfloat16, auto device map)...")
start_time = time.time()
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    subfolder=STEP,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Model loaded in {time.time() - start_time:.2f}s!")

In [ ]:
prompt = "วัดแก้วมรกตคืออะไร"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"\nPrompt: {prompt}")
print("="*50)

model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.3,
    no_repeat_ngram_size=4,
    streamer=streamer,
)
print("="*50)